# Tic Tac Toe with Reinforcement Learning

This notebook implements a Tic Tac Toe game where an AI player learns to play using reinforcement learning. The AI builds a value function through experience, learning which board states are more likely to lead to wins.

## Imports and Constants

First, we import necessary libraries and define our board dimensions.

In [ ]:
import numpy as np
import pickle

BOARD_ROWS = 3
BOARD_COLS = 3

## Game State Class

The `State` class represents the game board and controls game logic. It tracks:
- The current board configuration
- The two players
- Whether the game has ended
- The current player's turn
- A hash of the board state for reinforcement learning

It also includes methods for:
- Determining the winner
- Finding available positions
- Updating the board state
- Distributing rewards
- Training through self-play
- Playing against a human

In [ ]:
class State:
    def __init__(self, p1, p2):
        self.board = np.zeros((BOARD_ROWS, BOARD_COLS))
        self.p1 = p1
        self.p2 = p2
        self.isEnd = False
        self.boardHash = None
        # init p1 plays first
        self.playerSymbol = 1

    # get unique hash of current board state
    def getHash(self):
        self.boardHash = str(self.board.reshape(BOARD_COLS * BOARD_ROWS))
        return self.boardHash

    def winner(self):
        # row
        for i in range(BOARD_ROWS):
            if sum(self.board[i, :]) == 3:
                self.isEnd = True
                return 1
            if sum(self.board[i, :]) == -3:
                self.isEnd = True
                return -1
        # col
        for i in range(BOARD_COLS):
            if sum(self.board[:, i]) == 3:
                self.isEnd = True
                return 1
            if sum(self.board[:, i]) == -3:
                self.isEnd = True
                return -1
        # diagonal
        diag_sum1 = sum([self.board[i, i] for i in range(BOARD_COLS)])
        diag_sum2 = sum([self.board[i, BOARD_COLS - i - 1] for i in range(BOARD_COLS)])
        diag_sum = max(abs(diag_sum1), abs(diag_sum2))
        if diag_sum == 3:
            self.isEnd = True
            if diag_sum1 == 3 or diag_sum2 == 3:
                return 1
            else:
                return -1

        # tie
        # no available positions
        if len(self.availablePositions()) == 0:
            self.isEnd = True
            return 0
        # not end
        self.isEnd = False
        return None

    def availablePositions(self):
        positions = []
        for i in range(BOARD_ROWS):
            for j in range(BOARD_COLS):
                if self.board[i, j] == 0:
                    positions.append((i, j))  # need to be tuple
        return positions

    def updateState(self, position):
        self.board[position] = self.playerSymbol
        # switch to another player
        self.playerSymbol = -1 if self.playerSymbol == 1 else 1

    # only when game ends
    def giveReward(self):
        result = self.winner()
        # backpropagate reward
        if result == 1:
            self.p1.feedReward(1)
            self.p2.feedReward(0)
        elif result == -1:
            self.p1.feedReward(0)
            self.p2.feedReward(1)
        else:
            self.p1.feedReward(0.1)
            self.p2.feedReward(0.5)

    # board reset
    def reset(self):
        self.board = np.zeros((BOARD_ROWS, BOARD_COLS))
        self.boardHash = None
        self.isEnd = False
        self.playerSymbol = 1

    def play(self, rounds=100):
        for i in range(rounds):
            if i % 1000 == 0:
                print("Rounds {}".format(i))
            while not self.isEnd:
                # Player 1
                positions = self.availablePositions()
                p1_action = self.p1.chooseAction(positions, self.board, self.playerSymbol)
                # take action and upate board state
                self.updateState(p1_action)
                board_hash = self.getHash()
                self.p1.addState(board_hash)
                # check board status if it is end

                win = self.winner()
                if win is not None:
                    # self.showBoard()
                    # ended with p1 either win or draw
                    self.giveReward()
                    self.p1.reset()
                    self.p2.reset()
                    self.reset()
                    break

                else:
                    # Player 2
                    positions = self.availablePositions()
                    p2_action = self.p2.chooseAction(positions, self.board, self.playerSymbol)
                    self.updateState(p2_action)
                    board_hash = self.getHash()
                    self.p2.addState(board_hash)

                    win = self.winner()
                    if win is not None:
                        # self.showBoard()
                        # ended with p2 either win or draw
                        self.giveReward()
                        self.p1.reset()
                        self.p2.reset()
                        self.reset()
                        break

    # play with human
    def play2(self):
        while not self.isEnd:
            # Player 1
            positions = self.availablePositions()
            p1_action = self.p1.chooseAction(positions, self.board, self.playerSymbol)
            # take action and upate board state
            self.updateState(p1_action)
            self.showBoard()
            # check board status if it is end
            win = self.winner()
            if win is not None:
                if win == 1:
                    print(self.p1.name, "wins!")
                else:
                    print("tie!")
                self.reset()
                break

            else:
                # Player 2
                positions = self.availablePositions()
                p2_action = self.p2.chooseAction(positions)

                self.updateState(p2_action)
                self.showBoard()
                win = self.winner()
                if win is not None:
                    if win == -1:
                        print(self.p2.name, "wins!")
                    else:
                        print("tie!")
                    self.reset()
                    break

    def showBoard(self):
        # p1: x  p2: o
        for i in range(0, BOARD_ROWS):
            print('-------------')
            out = '| '
            for j in range(0, BOARD_COLS):
                if self.board[i, j] == 1:
                    token = 'x'
                if self.board[i, j] == -1:
                    token = 'o'
                if self.board[i, j] == 0:
                    token = ' '
                out += token + ' | '
            print(out)
        print('-------------')

## AI Player Class

The `Player` class represents an AI player that learns using reinforcement learning:
- Maintains a value function for different board states
- Uses an epsilon-greedy policy for exploration/exploitation
- Chooses actions based on expected future rewards
- Updates its policy after each game through backpropagation

In [ ]:
class Player:
    def __init__(self, name, exp_rate=0.3):
        self.name = name
        self.states = []  # record all positions taken
        self.lr = 0.2
        self.exp_rate = exp_rate
        self.decay_gamma = 0.9
        self.states_value = {}  # state -> value

    def getHash(self, board):
        boardHash = str(board.reshape(BOARD_COLS * BOARD_ROWS))
        return boardHash

    def chooseAction(self, positions, current_board, symbol):
        if np.random.uniform(0, 1) <= self.exp_rate:
            # take random action
            idx = np.random.choice(len(positions))
            action = positions[idx]
        else:
            value_max = -999
            for p in positions:
                next_board = current_board.copy()
                next_board[p] = symbol
                next_boardHash = self.getHash(next_board)
                value = 0 if self.states_value.get(next_boardHash) is None else self.states_value.get(next_boardHash)
                # print("value", value)
                if value >= value_max:
                    value_max = value
                    action = p
        # print("{} takes action {}".format(self.name, action))
        return action

    # append a hash state
    def addState(self, state):
        self.states.append(state)

    # at the end of game, backpropagate and update states value
    def feedReward(self, reward):
        for st in reversed(self.states):
            if self.states_value.get(st) is None:
                self.states_value[st] = 0
            self.states_value[st] += self.lr * (self.decay_gamma * reward - self.states_value[st])
            reward = self.states_value[st]

    def reset(self):
        self.states = []

    def savePolicy(self):
        fw = open('policy_' + str(self.name), 'wb')
        pickle.dump(self.states_value, fw)
        fw.close()

    def loadPolicy(self, file):
        fr = open(file, 'rb')
        self.states_value = pickle.load(fr)
        fr.close()

## Human Player Class

The `HumanPlayer` class represents a human player interacting with the game:
- Takes input from the console to determine moves
- Implements the same interface as the AI player for compatibility
- Doesn't need to learn or maintain a policy

In [ ]:
class HumanPlayer:
    def __init__(self, name):
        self.name = name

    def chooseAction(self, positions):
        while True:
            row = int(input("Input your action row:"))
            col = int(input("Input your action col:"))
            action = (row, col)
            if action in positions:
                return action

    # append a hash state
    def addState(self, state):
        pass

    # at the end of game, backpropagate and update states value
    def feedReward(self, reward):
        pass

    def reset(self):
        pass

## Main Execution

This section contains the main execution logic:
1. First, it trains two AI players against each other for 50,000 rounds
2. The learned policies are saved to files
3. Then, a human player can play against the trained AI

### First, play against the AI without training

In [ ]:
# Assume p1 is computer, p2 is human
p1 = Player("computer", exp_rate=0)
p2 = HumanPlayer("human")

# initialize state
st = State(p1, p2)

# Play
st.play2()

### Train the AI for 50,000 rounds

In [ ]:
# Assume p1 is computer, p2 is computer
p1 = Player("p1")
p2 = Player("p2")

# initialize state
st = State(p1, p2)

# Play
print("training...")
st.play(50000)

# Save policies
p1.savePolicy()
p2.savePolicy()

### Last, play against the AI after training to see the difference

In [ ]:
# Assume p1 is computer, p2 is human
p1 = Player("computer", exp_rate=0)
p2 = HumanPlayer("human")

# load trained p1 policy
p1.loadPolicy("policy_p1")

# initialize state
st = State(p1, p2)

# Play
st.play2()

## Explanation of the Reinforcement Learning Algorithm

This implementation uses a value-based reinforcement learning approach called Temporal Difference (TD) learning. Here's how it works:

1. **State Representation**: Each board configuration is represented as a string hash.

2. **Value Function**: The AI maintains a dictionary mapping state hashes to expected rewards.

3. **Exploration vs. Exploitation**: 
   - With probability `exp_rate`, the AI takes a random action (exploration).
   - Otherwise, it takes the action leading to the highest-valued state (exploitation).

4. **Learning Process**:
   - After each game, rewards are backpropagated through all visited states.
   - The value of each state is updated using the formula:
     ```
     V(s) = V(s) + lr * (gamma * V(s') - V(s))
     ```
     where `lr` is the learning rate, `gamma` is the discount factor, and `s'` is the next state.

5. **Reward Structure**:
   - Win: 1.0
   - Loss: 0.0
   - Draw: 0.1 for player 1, 0.5 for player 2

Over many training games, the AI learns to associate board states with their expected outcomes, gradually improving its play strategy.